# Module 2 — Student Diagnostic Analyzer

In [ ]:
!pip install google-generativeai -q

In [ ]:
import google.generativeai as genai
import json

GEMINI_API_KEY = ""
MODEL_NAME     = "gemini-2.0-flash"

genai.configure(api_key=GEMINI_API_KEY)

In [ ]:
with open('assessment_fractions.json', 'r', encoding='utf-8') as f:
    assessment_json = json.load(f)

In [ ]:
student_quiz_data = [
    {"question_id": 1,  "selected_option": "A", "time_taken_seconds": 20},
    {"question_id": 2,  "selected_option": "C", "time_taken_seconds": 35},
    {"question_id": 3,  "selected_option": "B", "time_taken_seconds": 58},
    {"question_id": 4,  "selected_option": "A", "time_taken_seconds": 42},
    {"question_id": 5,  "selected_option": "D", "time_taken_seconds": 90},
    {"question_id": 6,  "selected_option": "B", "time_taken_seconds": 75},
    {"question_id": 7,  "selected_option": "A", "time_taken_seconds": 120},
    {"question_id": 8,  "selected_option": "C", "time_taken_seconds": 95},
    {"question_id": 9,  "selected_option": "D", "time_taken_seconds": 110},
    {"question_id": 10, "selected_option": "B", "time_taken_seconds": 140}
]

q_map = {q['id']: q for q in assessment_json['questions']}
correct = sum(1 for r in student_quiz_data
              if q_map.get(r['question_id'], {}).get('correct_option') == r['selected_option'])
print(correct)

In [ ]:
def build_diagnostic_prompt(topic, student_quiz_data, assessment_json):
    q_map = {q['id']: q for q in assessment_json['questions']}

    response_block_lines = []
    for resp in student_quiz_data:
        q = q_map.get(resp['question_id'])
        if not q:
            continue
        is_correct = resp['selected_option'] == q['correct_option']
        chosen_text  = q['options'][resp['selected_option']]
        correct_text = q['options'][q['correct_option']]
        time_line = (f"  Time taken: {resp.get('time_taken_seconds')}s "
                     f"(estimated: {q['estimated_time_seconds']}s)"
                     if resp.get('time_taken_seconds') else '')

        block = [
            f"Question {q['id']} [{q['difficulty']}] — {q['micro_skill']}",
            f"  Q: {q['question']}",
            f"  Student answered: {resp['selected_option']}) {chosen_text} → "
            f"{'CORRECT' if is_correct else 'INCORRECT'}",
        ]
        if not is_correct:
            block.append(f"  Correct answer: {q['correct_option']}) {correct_text}")
        block += [
            f"  Misconception probed: {q['misconception_tested']}",
            f"  Bloom's: {q['blooms_level']} | Category: {q['concept_category']}",
        ]
        if time_line:
            block.append(time_line)
        response_block_lines.append('\n'.join(block))

    response_block = '\n\n'.join(response_block_lines)

    total = len(student_quiz_data)
    correct_count = sum(
        1 for r in student_quiz_data
        if q_map.get(r['question_id'], {}).get('correct_option') == r['selected_option']
    )

    return f"""You are an expert educational diagnostician specializing in learning analytics,
cognitive science, misconception detection, and mastery estimation.

Your task is to identify WHY the student answered incorrectly—not merely WHICH answers were incorrect.

Topic: {topic}
Subject: {assessment_json['subject']}
Grade: {assessment_json['grade']}
Raw Score: {correct_count}/{total} correct

Student Responses (with full question context):
{response_block}

WEIGHTING RULES:
- Hard questions = 3 points, Medium = 2 points, Easy = 1 point
- Repeated mistakes on the same micro-skill = elevated severity
- Do NOT simply count wrong answers — diagnose the cognitive gap

ROOT CAUSE — use exactly one of:
  "Concept missing" | "Calculation error" | "Reading misunderstanding"
  "Guessing" | "Carelessness" | "Pattern confusion" | "Unknown"

MASTERY SCORING (weighted):
  90-100 → Advanced
  70-89  → Intermediate
  40-69  → Beginner
  0-39   → Needs Immediate Intervention

Return ONLY valid JSON. No markdown. No explanations outside JSON.

{{
  "overall_score": 0,
  "mastery_level": "",
  "strengths": [],
  "identified_gaps": [
    {{
      "micro_skill": "",
      "severity": "High | Medium | Low",
      "misconception": "",
      "root_cause": "",
      "evidence": ["Question 2", "Question 7"],
      "confidence": 0.96,
      "recommended_priority": "Critical | Recommended | Optional"
    }}
  ],
  "learning_path": [],
  "gap_summary": "",
  "teacher_notes": "",
  "student_friendly_summary": ""
}}"""


In [ ]:
def analyze_responses(topic, student_quiz_data, assessment_json):
    model = genai.GenerativeModel(
        model_name=MODEL_NAME,
        generation_config={
            'response_mime_type': 'application/json',
            'temperature': 0.2,
            'max_output_tokens': 4096,
        }
    )
    prompt = build_diagnostic_prompt(topic, student_quiz_data, assessment_json)
    response = model.generate_content(prompt)
    return json.loads(response.text)

In [ ]:
report = analyze_responses(
    topic            = assessment_json['topic'],
    student_quiz_data= student_quiz_data,
    assessment_json  = assessment_json,
)

print(report['overall_score'])
print(report['mastery_level'])
print(len(report['strengths']))
print(len(report['identified_gaps']))
print(len(report['learning_path']))

In [ ]:
print(report['overall_score'])
print(report['mastery_level'])

for s in report['strengths']:
    print(s)

for gap in report['identified_gaps']:
    print(gap['severity'], gap['micro_skill'], gap['recommended_priority'])
    print(gap['misconception'])
    print(gap['root_cause'])
    print(gap['evidence'])
    print(gap['confidence'])

for i, step in enumerate(report['learning_path'], 1):
    print(i, step)

print(report['gap_summary'])
print(report['teacher_notes'])
print(report['student_friendly_summary'])

In [ ]:
out_file = f"diagnostic_report_{assessment_json['topic'].lower().replace(' ', '_')}.json"
with open(out_file, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)